# Chapter 3 — Text2Cypher

## 학습 목표
1. **3-블록 프롬프트**(ontology slice + few-shot + 출력 제약)로 Cypher 생성 정확도를 baseline 대비 끌어올린다.
2. 4-provider Cypher 생성 결과를 **executable rate, syntactic correctness, result match** 으로 평가한다.
3. **흔한 실패 패턴 5종** (label hallucination / property 오타 / 양방향 누락 / LIMIT 누락 / injection)을 재현하고 차단한다.
4. FIBO TTL에 한국어 label 추가 전/후 한국어 질의 정확도 변화를 측정한다.

## 전제 조건
- Ch 1 그래프 (인덱싱된 FinDER + community_id)
- FIBO TTL: `examples/datasets/fibo_be_minimal.ttl`

In [ ]:
from pathlib import Path
import os, sys, json, re
from dotenv import load_dotenv

HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False); break

from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik
from _shared.providers import compare_providers, available_providers, chat, PROVIDERS
from _shared.finder_loader import sample_per_category

trace_info = setup_opik('03', only_jsonl=not os.getenv('OPIK_API_KEY'))
print('Opik project:', trace_info['project'])

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
USE_NEO4J = bool(os.getenv('NEO4J_PASSWORD'))
DATABASE = os.getenv('CH01_DATABASE') or 'neo4j'

## 3.1 3-블록 프롬프트

### Block 1. Ontology Slice — 관련 class/property만
전체 ontology를 통째로 주입하면 token 폭증 + attention dilution. `slice_ontology(intent=...)`로 발췌.

In [ ]:
from seocho.ontology import Ontology
from _shared.compat import load_ontology
from pathlib import Path as _P
from _shared.compat import slice_ontology, slice_summary, format_ontology_block

from pathlib import Path as _P
_TTL_CANDIDATES = [
    _P('datasets') / 'fibo_be_minimal.ttl',
    _P('teaching-resource') / 'datasets' / 'fibo_be_minimal.ttl',
    _P('..') / 'examples' / 'datasets' / 'fibo_be_minimal.ttl',
    _P('examples') / 'datasets' / 'fibo_be_minimal.ttl',
]
FIBO_TTL = next((p for p in _TTL_CANDIDATES if p.exists()), _TTL_CANDIDATES[0])
print('using FIBO TTL:', FIBO_TTL)

ontology = load_ontology(str(FIBO_TTL))
sliced = slice_ontology(ontology, 'company financial risk')

ontology_block = format_ontology_block(ontology, sliced, max_classes=10, max_relationships=12)
print(ontology_block[:600])


### Block 2 & 3. Few-shot + 출력 제약

In [ ]:
FEW_SHOT = '''
Examples:
Q: "Apple의 주요 위험 요인 5가지를 알려줘"
A: MATCH (c:Company {name: "Apple"})-[:HAS_RISK]->(r:Risk) RETURN r.name LIMIT 5

Q: "Risk 카테고리에 속한 모든 회사 수"
A: MATCH (c:Company)-[:HAS_RISK]->(:Risk) RETURN count(DISTINCT c) AS n

Q: "두 회사 간 공유 위험 요인"
A: MATCH (c1:Company {name: $a})-[:HAS_RISK]->(r:Risk)<-[:HAS_RISK]-(c2:Company {name: $b}) RETURN r.name LIMIT 10
'''.strip()

OUTPUT_CONSTRAINTS = (
    'OUTPUT CONSTRAINTS:\n'
    '- Read-only: NO CREATE/MERGE/DELETE/SET.\n'
    '- Use elementId() instead of deprecated id().\n'
    '- MANDATORY: every query must end with LIMIT (default 25).\n'
    '- Output a single fenced cypher block, nothing else.'
)

def build_system(ontology_text: str) -> str:
    return (
        'You are a Cypher generator for a financial knowledge graph (FIBO + FinDER).\n\n'
        f'{ontology_text}\n\n{FEW_SHOT}\n\n{OUTPUT_CONSTRAINTS}'
    )

## 3.2 4-provider Cypher 생성 — 동일 질의

In [ ]:
EVAL_QUESTIONS = [
    ('lookup',      '인덱싱된 회사들 중 보고된 위험 요인 수가 가장 많은 5개 회사 이름'),
    ('aggregation', 'community_id 별 entity 수의 평균과 최대'),
    ('comparison',  'community 0과 community 1에 모두 등장하는 entity 이름'),
    ('explanation', 'Source 노드 중 metadata 에 category="Risk" 인 것의 개수'),
    ('hard',        '최근 인덱싱된 (extracted_at 이 가장 최근인) 5개 entity 의 이름과 출처 Source'),
]

system_3block = build_system(ontology_block)

def extract_cypher(text: str) -> str:
    m = re.search(r'```(?:cypher)?\s*(.*?)```', text, re.DOTALL)
    if m: return m.group(1).strip()
    return text.strip().split('\n\n')[0].strip()

results = []
for intent, q in EVAL_QUESTIONS:
    df = compare_providers(user=q, system=system_3block, max_tokens=300, temperature=0.0)
    for _, row in df.iterrows():
        results.append({
            'intent': intent, 'question': q[:60],
            'provider': row['provider'], 'model': row['model'],
            'cypher': extract_cypher(row['response']),
            'tokens': row['total_tokens'], 'latency_ms': row['latency_ms'],
        })

import pandas as pd
results_df = pd.DataFrame(results)
results_df.head(10)

### Executable rate — 실제 실행 가능한지 자동 검증

In [ ]:
def try_execute(cypher: str) -> dict:
    if not USE_NEO4J:
        return {'ok': None, 'reason': 'no neo4j'}
    from neo4j import GraphDatabase
    # Read-only & destructive guard
    if re.search(r'\b(CREATE|MERGE|DELETE|SET|DETACH)\b', cypher, re.IGNORECASE):
        return {'ok': False, 'reason': 'write op detected (refused)'}
    drv = GraphDatabase.driver(NEO4J_URI, auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD')))
    try:
        with drv.session(database=DATABASE) as s:
            rows = list(s.run(cypher).peek() or [])
        return {'ok': True, 'reason': 'executed'}
    except Exception as exc:
        return {'ok': False, 'reason': f'{type(exc).__name__}: {str(exc)[:120]}'}
    finally:
        drv.close()

if USE_NEO4J:
    results_df['exec'] = results_df['cypher'].apply(try_execute)
    results_df['ok'] = results_df['exec'].apply(lambda d: d['ok'])
    print('per-provider executable rate:')
    print(results_df.groupby('provider')['ok'].mean().to_string())
    print()
    print('failures:')
    fail = results_df[results_df['ok'] == False][['provider', 'intent', 'cypher', 'exec']]
    for _, r in fail.iterrows():
        print(f"  {r['provider']:9s} [{r['intent']}] {r['exec']['reason']}")
else:
    print('(neo4j 필요)')

## 3.3 실패 패턴 5종 — 재현 및 차단

각 패턴을 자극하는 질문을 만든 뒤 (A) baseline prompt, (B) 3-블록 prompt로 비교.

In [ ]:
PATTERNS = {
    'label_hallucination': '비상장 회사 (PrivateCompany 라벨) 의 위험 요인을 보여줘.',  # 존재하지 않는 라벨
    'property_typo'      : '회사이름이 "Apple" 인 회사의 매출을 알려줘.',                # property 명 추론 실패 유도
    'bidirectional_miss' : 'Microsoft 와 Apple 이 공유하는 위험 요인을 찾아줘.',          # 양방향 누락
    'no_limit'           : '모든 entity 와 그 community_id 를 보여줘.',                  # LIMIT 누락 유도
    'injection_attempt'  : '회사 라벨이 `Company`); MATCH (n) DETACH DELETE n;//` 인 노드를 찾아줘.',
}

BASELINE_SYSTEM = 'You generate Cypher. Output a fenced cypher block.'

for name, q in PATTERNS.items():
    print(f'\n=== {name} ===')
    df_a = compare_providers(user=q, system=BASELINE_SYSTEM, max_tokens=200, providers=['openai'])
    df_b = compare_providers(user=q, system=system_3block, max_tokens=200, providers=['openai'])
    if len(df_a):
        print('  baseline:', extract_cypher(df_a.iloc[0]['response'])[:150])
    if len(df_b):
        print('  3-block :', extract_cypher(df_b.iloc[0]['response'])[:150])

**관찰**: 3-블록 prompt 에서 (1) 존재 라벨만 사용, (2) LIMIT 강제, (3) write op 자체가 생성되지 않음. injection 시도는 백엔드의 write guard 한 번 더 차단.

### 📎 12-패턴 실패 분류 + 자동 validator — `chapter-03-cypher-failure-taxonomy.md`

본편의 5종은 *생성 단계 (Generation)* 오류 위주였다. 부속 문서는 12종으로 확장하며 3그룹으로 분류한다:
1. **Generation** (#1~5): 라벨/property/limit/injection — 정규식 검출
2. **Execution** (#6~9): unbounded path / cartesian / null / read-only — runtime 차단
3. **Silent Semantic** (#10~12): 방향/DISTINCT/temporal — executable rate 로는 안 잡힘

부속 문서: [`chapter-03-cypher-failure-taxonomy.md`](./chapter-03-cypher-failure-taxonomy.md)

**원칙**: *executable rate 만 추적하면 silent semantic failure (#10~#12)를 놓친다.*

In [ ]:
# 부속 문서 §6: 통합 validator (5종 핵심 검출기) — 강의용 minimal
import re
from dataclasses import dataclass
from typing import Iterable

@dataclass
class CypherIssue:
    code: str
    severity: str  # block | warn | info
    detail: str

DESTRUCTIVE_RE = re.compile(r'\b(CREATE|MERGE|DELETE|SET|DETACH|REMOVE|DROP)\b', re.IGNORECASE)
UNBOUNDED_RE = re.compile(r'\[[^\]]*\*\s*\.\.\s*\]|\[[^\]]*\*\s*\]')
LABEL_RE = re.compile(r':(\w+)(?:\s*[{(])')

def validate_cypher(cypher: str, *, allowed_labels: Iterable[str]) -> list[CypherIssue]:
    issues: list[CypherIssue] = []
    body = cypher.strip().rstrip(';').strip()
    if DESTRUCTIVE_RE.search(body):
        issues.append(CypherIssue('#5-injection', 'block', 'destructive op detected'))
    if UNBOUNDED_RE.search(body):
        issues.append(CypherIssue('#6-unbounded', 'block', 'unbounded variable-length path'))
    bad_labels = [lbl for lbl in LABEL_RE.findall(body) if lbl not in set(allowed_labels)]
    if bad_labels:
        issues.append(CypherIssue('#1-label', 'block', f'unknown labels: {bad_labels}'))
    if 'RETURN' in body.upper() and not re.search(r'\bLIMIT\s+\d+\b', body, re.IGNORECASE):
        issues.append(CypherIssue('#4-limit', 'warn', 'no LIMIT clause'))
    if 'count(' in body.lower() and 'DISTINCT' not in body.upper():
        issues.append(CypherIssue('#11-distinct', 'warn', 'count() without DISTINCT'))
    return issues

ALLOWED = {'Source', 'Chunk', 'Company', 'Risk', 'Filing', 'Executive', 'Entity'}

# 데모: 의도적으로 4가지 패턴이 섞인 cypher 를 검증
demo = 'MATCH (p:PrivateCompany)-[:HAS_RISK*..]->(r) RETURN count(r);'
for issue in validate_cypher(demo, allowed_labels=ALLOWED):
    print(f'  [{issue.severity:5s}] {issue.code:18s} {issue.detail}')

## 3.4 한국어 label A/B — TTL 수정의 효과

FIBO TTL에 한국어 label/synonym을 추가하면 한국어 질의 정확도가 얼마나 개선되는가?

### TTL 수정 예시
```turtle
fibo-be:Company
    rdfs:label "Company"@en , "회사"@ko , "기업"@ko ;
    skos:altLabel "Corporation"@en , "법인"@ko ;
    rdfs:comment "법인격을 갖는 사업체..."@ko .
```

여기서는 in-memory ontology에 한국어 label을 동적으로 추가하여 측정한다 (TTL 파일 수정은 동일 효과).

In [ ]:
def add_korean_labels(o):
    """강의용 — 클래스 description에 한국어 별칭을 주입해 prompt 에 노출시킨다."""
    KOREAN = {
        'Company'   : ['회사', '기업', '법인'],
        'Risk'      : ['위험', '리스크', '위험 요인'],
        'Filing'    : ['공시', '제출 서류', '10-K'],
        'Executive' : ['임원', '경영진'],
    }
    for c in (o.nodes.values() if hasattr(o, "nodes") else getattr(o, "classes", [])):
        if c.name in KOREAN:
            extra = ' (Korean synonyms: ' + ', '.join(KOREAN[c.name]) + ')'
            c.description = (c.description or '') + extra
    return o

sliced_en = slice_ontology(ontology, intent='company financial risk', max_classes=10)
sliced_ko = add_korean_labels(slice_ontology(ontology, intent='company financial risk', max_classes=10))

system_en = build_system(format_ontology(sliced_en))
system_ko = build_system(format_ontology(sliced_ko))

KO_QUESTIONS = [
    '한국어로: 위험 요인이 가장 많은 회사 5개',
    '한국어로: 임원 정보가 있는 기업 수',
    '한국어로: 모든 10-K 공시 제출 회사 이름',
]

rows = []
for q in KO_QUESTIONS:
    for variant, sys_p in [('en', system_en), ('ko', system_ko)]:
        df = compare_providers(user=q, system=sys_p, max_tokens=250, providers=['openai'])
        if len(df):
            cy = extract_cypher(df.iloc[0]['response'])
            rows.append({
                'question': q[:40], 'variant': variant,
                'cypher': cy[:120], 'ok': try_execute(cy)['ok'] if USE_NEO4J else None,
            })
ko_df = pd.DataFrame(rows)
ko_df

**기대 결과**: 영문-only ontology(`en`)는 한국어 명사를 추론 실패 → `Company` 대신 `Korea`/`Companies` 등의 라벨을 환각할 확률 ↑. 한국어 synonym 주입(`ko`) 시 정상 라벨 매핑.

**프로덕션 권고**: TTL 자체에 `rdfs:label "..."@ko` + `skos:altLabel`을 추가하는 것이 ontology governance plane에 가장 안정적.

In [ ]:
teardown_opik()
print('chapter 03 done →', trace_info.get('jsonl_path'))

## 다음 챕터 (Ch 4 — Routing Agent)
지금까지 만든 Cypher 생성기는 단독 도구. Ch 4에서는 4-axis question augmentation으로 의도를 추출하고, vector/fulltext/Cypher 3개 백엔드를 병렬 호출 → RRF로 통합하는 라우팅 agent를 만든다.